In [2]:
data = [("me gusta comer en la cafeteria en la mañana".split(), "SPANISH"),
        ("Give it to me".split(), "ENGLISH"),
        ("No creo que sea una buena idea".split(), "SPANISH"),
        ("No it is not a good idea to get lost at sea".split(), "ENGLISH")]
test_data = [("Yo creo que si".split(), "SPANISH"),
             ("it is lost on me".split(), "ENGLISH")]
word_to_index = {}
for sentence, _ in data + test_data:
    for word in sentence:
        if word not in word_to_index:
            word_to_index[word] = len(word_to_index)
vocab_size = len(word_to_index)
num_labels = 2
print("Vocab size:", vocab_size)
print("Word to index mapping:", word_to_index)

Vocab size: 27
Word to index mapping: {'me': 0, 'gusta': 1, 'comer': 2, 'en': 3, 'la': 4, 'cafeteria': 5, 'mañana': 6, 'Give': 7, 'it': 8, 'to': 9, 'No': 10, 'creo': 11, 'que': 12, 'sea': 13, 'una': 14, 'buena': 15, 'idea': 16, 'is': 17, 'not': 18, 'a': 19, 'good': 20, 'get': 21, 'lost': 22, 'at': 23, 'Yo': 24, 'si': 25, 'on': 26}


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
class BoWClassifier(nn.Module):
    def __init__(self, num_labels, vocab_size):
        super(BoWClassifier, self).__init__()

        self.linear = nn.Linear(vocab_size, num_labels)

    def forward(self, bow_vec):
        return F.log_softmax(self.linear(bow_vec), dim=1)

def make_bow_vector(sentence, word_to_index):
    vec = torch.zeros(len(word_to_index))
    for word in sentence:
        if word in word_to_index:
            index = word_to_index[word]
            vec[index] += 1
    return vec.view(1, -1)

def make_target(label, label_to_index):
    return torch.LongTensor([label_to_index[label]])

In [12]:
model = BoWClassifier(num_labels, vocab_size)

for param in model.parameters():
    print(param)

with torch.no_grad():
    sample = data[0]
    print("Sample sentence:", sample[0])
    print("Wor d to index mapping:", word_to_index)
    bow_vec = make_bow_vector(sample[0], word_to_index)
    print("BoW vector for sample:", bow_vec)
    log_probs = model(bow_vec)
    print("Log probabilities:", log_probs)
        

Parameter containing:
tensor([[-0.0942, -0.0371, -0.1355, -0.0895,  0.0940,  0.0111, -0.1134,  0.1593,
         -0.0768, -0.1095, -0.0325, -0.0196,  0.0629, -0.1236,  0.1435, -0.1511,
         -0.1853,  0.0730, -0.1895, -0.0691,  0.1240,  0.0309, -0.1303,  0.1077,
         -0.0342, -0.0067,  0.1729],
        [ 0.1655,  0.1579, -0.0506,  0.0976,  0.1220,  0.1225, -0.0591, -0.0229,
         -0.1543,  0.0362,  0.0398, -0.1354, -0.0299,  0.0951,  0.0194, -0.1020,
         -0.0309, -0.1059, -0.1654,  0.1472, -0.1155,  0.1467,  0.0353,  0.1105,
         -0.0782,  0.1397,  0.1075]], requires_grad=True)
Parameter containing:
tensor([0.0735, 0.0718], requires_grad=True)
Sample sentence: ['me', 'gusta', 'comer', 'en', 'la', 'cafeteria', 'en', 'la', 'mañana']
Wor d to index mapping: {'me': 0, 'gusta': 1, 'comer': 2, 'en': 3, 'la': 4, 'cafeteria': 5, 'mañana': 6, 'Give': 7, 'it': 8, 'to': 9, 'No': 10, 'creo': 11, 'que': 12, 'sea': 13, 'una': 14, 'buena': 15, 'idea': 16, 'is': 17, 'not': 18, 'a': 1

In [13]:
lavel_to_index = {"ENGLISH": 0, "SPANISH": 1}

with torch.no_grad():
    for sentence, label in test_data:
        bow_vec = make_bow_vector(sentence, word_to_index)
        log_probs = model(bow_vec)
        print(f"Sentence: {sentence}")
        print(f"Label: {label}")
        print(f"Log probabilities: {log_probs}")

Sentence: ['Yo', 'creo', 'que', 'si']
Label: SPANISH
Log probabilities: tensor([[-0.6406, -0.7486]])
Sentence: ['it', 'is', 'lost', 'on', 'me']
Label: ENGLISH
Log probabilities: tensor([[-0.7453, -0.6436]])


In [14]:
next(model.parameters())[:, word_to_index["creo"]]

tensor([-0.0196, -0.1354], grad_fn=<SelectBackward0>)

In [15]:
loss_fuction = nn.NLLLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [16]:
for epoch in range(100):
    for sentence, label in data:
        model.zero_grad()

        bow_vec = make_bow_vector(sentence, word_to_index)
        target = make_target(label, lavel_to_index)

        log_probs = model(bow_vec)
        loss = loss_fuction(log_probs, target)
        loss.backward()
        optimizer.step()

with torch.no_grad():
    for sentence, label in test_data:
        bow_vec = make_bow_vector(sentence, word_to_index)
        log_probs = model(bow_vec)
        print(f"Sentence: {sentence}")
        print(f"Label: {label}")
        print(f"Log probabilities: {log_probs}")

Sentence: ['Yo', 'creo', 'que', 'si']
Label: SPANISH
Log probabilities: tensor([[-1.7228, -0.1967]])
Sentence: ['it', 'is', 'lost', 'on', 'me']
Label: ENGLISH
Log probabilities: tensor([[-0.0392, -3.2581]])


In [17]:
next(model.parameters())[:, word_to_index["creo"]]

tensor([-0.4732,  0.3182], grad_fn=<SelectBackward0>)